# Knowledge Refinement

## Why Knowledge Refinement Needed?
When documents are stored in vector databases, text splitting/chunking is usually performed based on fixed character counts (e.g., 900 characters) rather than semantic topic boundaries.

Consequently:
- A retrieved chunk may contain a relevant answer along with irrelevant text from adjacent topics or paragraphs.
- Passing raw, noisy chunks to the LLM dilutes the prompt context and reduces the overall quality of the generated response.

## The 3-Step Knowledge Refinement Pipeline

Knowledge Refinement transforms raw document chunks into clean context through three sequential steps

- **Decomposition:** The retrieved document chunk is split into fine-grained "sentence strips" ($S_1, S_2, \dots, S_k$) at the individual sentence level
- **Filtration:**
    - Each sentence strip is evaluated individually against the user query using a strict relevance filter
    - The filter determines if the sentence strip directly helps answer the query, assigning a decision (`keep = True` or `keep = False`)
- **Re-combination:**
    - All strips marked `keep = True` are concatenated back together into a compact, noise-free context block (refined_context)
    - Strips marked `keep = False` are discarded.


The **Knowledge Refinement** process is illustrated using an example where a retrieved document chunk contains both relevant and off-topic sentences due to fixed character-based chunking.

---

### Example Setup

* **User Question (\\(q\\))**: **"What is gradient descent?"**
* **Raw Retrieved Document Chunk (\\(d_1\\))**:
  > *"Gradient descent is an optimization algorithm used to minimize a loss function. It iteratively updates parameters in the direction of negative gradient. Neural networks are composed of layers of neurons with non-linear functions. Convolutional neural networks are particularly effective for image processing tasks."*

**The Problem:** The first two sentences accurately describe gradient descent, but the last two sentences discuss neural network layers and CNNs. This noise occurred because fixed text chunking (e.g., 900 characters) captured trailing text from an adjacent paragraph.

> Note: Depending upon the set $k$ value we might be getting $k$ number of documents such as $d_1, d_2, \dots, d_k$ and the same 3-step knowledge refinement will be applied to each of the documents. Here, we are just considering one retrieved document chunk for illustration.
---

### The 3-Step Knowledge Refinement Process

#### Step 1: Decomposition (Creating Sentence Strips)
The raw document \\(d_1\\) is split at sentence boundaries into individual **sentence strips** (\\(S_1, S_2, S_3, S_4\\)):

* **\\(S_1\\)**: *"Gradient descent is an optimization algorithm used to minimize a loss function."*
* **\\(S_2\\)**: *"It iteratively updates parameters in the direction of negative gradient."*
* **\\(S_3\\)**: *"Neural networks are composed of layers of neurons with non-linear functions."*
* **\\(S_4\\)**: *"Convolutional neural networks are particularly effective for image processing tasks."*

---

#### Step 2: Filtration (Scoring Each Strip)
Each sentence strip is evaluated against the query \\(q\\) using a strict relevance filter (such as a fine-tuned T5-Large model or an LLM):

* **\\(S_1\\)** \\(\rightarrow\\) Relevant to gradient descent \\(\rightarrow\\) **Keep (`True`)**
* **\\(S_2\\)** \\(\rightarrow\\) Relevant to gradient descent \\(\rightarrow\\) **Keep (`True`)**
* **\\(S_3\\)** \\(\rightarrow\\) Irrelevant (general neural network architecture) \\(\rightarrow\\) **Discard (`False`)**
* **\\(S_4\\)** \\(\rightarrow\\) Irrelevant (CNN image processing) \\(\rightarrow\\) **Discard (`False`)**

> Note: The fine-tuned T5-Large model is used by the authors in the paper: [Corrective Retrieval Augmented Generation](https://arxiv.org/abs/2401.15884)
---

#### Step 3: Re-combination (Refined Context)
The strips marked `keep = True` (\\(S_1\\) and \\(S_2\\)) are concatenated back together to create the final **refined document context**:

> **Refined \\(d_1\\) Context:**
> *"Gradient descent is an optimization algorithm used to minimize a loss function. It iteratively updates parameters in the direction of negative gradient."*

This clean, noise-free context is then passed to the LLM during answer generation, ensuring higher accuracy and preventing context pollution.